Notebook for first attempt to build a regression of football match results based on the 

In [2]:
# Load the necessary python packages
import pandas as pd
import sklearn

In [20]:
# Load the Premier League datasets from the directory football_data_uk
data_dict = {}
df_pl_2017 = pd.read_csv('football_data_uk/premier_league_results_2017.csv')
data_dict['2017']=df_pl_2017
df_pl_2018 = pd.read_csv('football_data_uk/premier_league_results_2018.csv')
data_dict['2018']=df_pl_2018
df_pl_2019 = pd.read_csv('football_data_uk/premier_league_results_2019.csv')
data_dict['2019']=df_pl_2019
df_pl_2020 = pd.read_csv('football_data_uk/premier_league_results_2020.csv')
data_dict['2020']=df_pl_2020
df_pl_2021 = pd.read_csv('football_data_uk/premier_league_results_2021.csv')
data_dict['2021']=df_pl_2021
df_pl_2022 = pd.read_csv('football_data_uk/premier_league_results_2022.csv')
data_dict['2022']=df_pl_2022
df_pl_2023 = pd.read_csv('football_data_uk/premier_league_results_2023.csv')
data_dict['2023']=df_pl_2023
df_pl_2024 = pd.read_csv('football_data_uk/premier_league_results_2024.csv')
data_dict['2024']=df_pl_2024
df_pl_2025 = pd.read_csv('football_data_uk/premier_league_results_2025.csv')
data_dict['2025']=df_pl_2025
df_pl_2026 = pd.read_csv('football_data_uk/premier_league_results_2026.csv')
data_dict['2026']=df_pl_2026

Before separating the datasets in training, validation and testing, let's investigate the content of one.

In [21]:
# Display the contents of the oldest dataset from the Premier League season 2017-2018
df_pl_2017.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
0,E0,11/08/2017,Arsenal,Leicester,4,3,H,2,2,D,...,2.32,21,-1.00,1.91,1.85,2.10,2.02,1.49,4.73,7.25
1,E0,12/08/2017,Brighton,Man City,0,2,A,0,0,D,...,2.27,20,1.50,1.95,1.91,2.01,1.96,11.75,6.15,1.29
2,E0,12/08/2017,Chelsea,Burnley,2,3,A,0,3,A,...,2.23,20,-1.75,2.03,1.97,1.95,1.90,1.33,5.40,12.25
3,E0,12/08/2017,Crystal Palace,Huddersfield,0,3,A,0,2,A,...,1.72,18,-0.75,2.10,2.05,1.86,1.83,1.79,3.56,5.51
4,E0,12/08/2017,Everton,Stoke,1,0,H,1,0,H,...,1.76,19,-0.75,1.94,1.90,2.01,1.98,1.82,3.49,5.42


There are 65 columns to the dataset. Let's see whether they are all neccesary or relevant to what

In [22]:
base_cols = df_pl_2017.columns
print(len(base_cols))
print(base_cols)

65
Index(['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG',
       'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC',
       'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD',
       'BWA', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA',
       'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'Bb1X2', 'BbMxH', 'BbAvH',
       'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5',
       'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH',
       'BbMxAHA', 'BbAvAHA', 'PSCH', 'PSCD', 'PSCA'],
      dtype='object')


In [23]:
for year, df in data_dict.items():
    if len(df.columns) < 65:
        print(year, ' has fewer columns:', len(df.columns))
    elif len(df.columns) > 65:
        print(year, ' has more columns', len(df.columns))

2018  has fewer columns: 62
2019  has more columns 106
2020  has more columns 106
2021  has more columns 106
2022  has more columns 106
2023  has more columns 106
2024  has more columns 120
2025  has more columns 132
2026  has more columns 114


Based on the notes from the website (https://football-data.co.uk/notes.txt), the respective columns are:

- Div = League Division
- Date = Match Date (dd/mm/yy)
- Time = Time of match kick off
- HomeTeam = Home Team
- AwayTeam = Away Team
- FTHG and HG = Full Time Home Team Goals
- FTAG and AG = Full Time Away Team Goals
- FTR and Res = Full Time Result (H=Home Win, D=Draw, A=Away Win)
- HTHG = Half Time Home Team Goals
- HTAG = Half Time Away Team Goals
- HTR = Half Time Result (H=Home Win, D=Draw, A=Away Win)

Match Statistics (where available)
- Attendance = Crowd Attendance
- HxG = Home Team expected goals
- AxG = Away Team expected goals
- HS = Home Team Shots
- AS = Away Team Shots
- HST = Home Team Shots on Target
- AST = Away Team Shots on Target
- HHW = Home Team Hit Woodwork
- AHW = Away Team Hit Woodwork
- HC = Home Team Corners
- AC = Away Team Corners
- HF = Home Team Fouls Committed
- AF = Away Team Fouls Committed
- HFKC = Home Team Free Kicks Conceded
- AFKC = Away Team Free Kicks Conceded
- HO = Home Team Offsides
- AO = Away Team Offsides
- HY = Home Team Yellow Cards
- AY = Away Team Yellow Cards
- HR = Home Team Red Cards
- AR = Away Team Red Cards
- HBP = Home Team Bookings Points (10 = yellow, 25 = red)
- ABP = Away Team Bookings Points (10 = yellow, 25 = red)

The others corresponds to odds provided by betting companies. Let's first remove these data points. 

In [24]:
rel_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG',
       'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC',
       'AC', 'HY', 'AY', 'HR', 'AR']
df_pl_2017_sel = df_pl_2017.loc[:, rel_cols]
df_pl_2017_sel.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,11/08/2017,Arsenal,Leicester,4,3,H,2,2,D,M Dean,...,10,3,9,12,9,4,0,1,0,0
1,12/08/2017,Brighton,Man City,0,2,A,0,0,D,M Oliver,...,2,4,6,9,3,10,0,2,0,0
2,12/08/2017,Chelsea,Burnley,2,3,A,0,3,A,C Pawson,...,6,5,16,11,8,5,3,3,2,0
3,12/08/2017,Crystal Palace,Huddersfield,0,3,A,0,2,A,J Moss,...,4,6,7,19,12,9,1,3,0,0
4,12/08/2017,Everton,Stoke,1,0,H,1,0,H,N Swarbrick,...,4,1,13,10,6,7,1,1,0,0


Proceed for all datasets.

In [25]:
data_dict_short = {}
for year, df in data_dict.items():
    print('Removing columns in year ', year)
    data_dict_short[year] = df.loc[:, rel_cols]

Removing columns in year  2017
Removing columns in year  2018
Removing columns in year  2019
Removing columns in year  2020
Removing columns in year  2021
Removing columns in year  2022
Removing columns in year  2023
Removing columns in year  2024
Removing columns in year  2025
Removing columns in year  2026


Verify that all dataset now have same number of columns (22).

In [28]:
for year, df in data_dict_short.items():
    if len(df.columns) < 22:
        print(year, ' has fewer columns:', len(df.columns))
    elif len(df.columns) > 22:
        print(year, ' has more columns', len(df.columns))
    else:
        print(year, ' has a similar number of columns.')

2017  has a similar number of columns.
2018  has a similar number of columns.
2019  has a similar number of columns.
2020  has a similar number of columns.
2021  has a similar number of columns.
2022  has a similar number of columns.
2023  has a similar number of columns.
2024  has a similar number of columns.
2025  has a similar number of columns.
2026  has a similar number of columns.


Now they can be concatenated. 

In [30]:
total_df = pd.concat(data_dict_short.values())

In [41]:
total_df.describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
count,3520.00000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000,3520.000000
mean,1.55000,1.285795,0.697727,0.565341,13.875852,11.511932,4.760227,4.037500,10.566193,10.816193,5.643466,4.677273,1.651989,1.858523,0.054830,0.063068
std,1.31479,1.202027,0.848646,0.779638,5.657928,5.066803,2.588125,2.352017,3.437444,3.563657,3.051493,2.794536,1.278796,1.314395,0.238648,0.248896
min,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.00000,0.000000,0.000000,0.000000,10.000000,8.000000,3.000000,2.000000,8.000000,8.000000,3.000000,3.000000,1.000000,1.000000,0.000000,0.000000
50%,1.00000,1.000000,0.000000,0.000000,13.000000,11.000000,4.000000,4.000000,10.000000,11.000000,5.000000,4.000000,2.000000,2.000000,0.000000,0.000000
75%,2.00000,2.000000,1.000000,1.000000,17.000000,15.000000,6.000000,5.000000,13.000000,13.000000,8.000000,6.000000,2.000000,3.000000,0.000000,0.000000
max,9.00000,9.000000,5.000000,5.000000,36.000000,37.000000,16.000000,15.000000,23.000000,26.000000,19.000000,19.000000,7.000000,8.000000,2.000000,2.000000


In [35]:
total_df.dtypes

Date        object
HomeTeam    object
AwayTeam    object
FTHG         int64
FTAG         int64
FTR         object
HTHG         int64
HTAG         int64
HTR         object
Referee     object
HS           int64
AS           int64
HST          int64
AST          int64
HF           int64
AF           int64
HC           int64
AC           int64
HY           int64
AY           int64
HR           int64
AR           int64
dtype: object

In [40]:
total_df.FTR.replace("H", "1")
total_df.FTR.replace("A", "-1")
total_df.FTR.replace("D", "0")

0     H
1     A
2     A
3     A
4     H
     ..
95    H
96    A
97    A
98    H
99    0
Name: FTR, Length: 3520, dtype: object